# 03. 정부식 10km 접근성·인구특성과 이용률 EDA
- 목적: H3SFCA 설계 전 단계에서, 구별 이용률 차이가 정부식 접근성보다 인구통계적 특성과 더 맞물리는지 확인함.
- 이용률: 2025년 구별 문화누리 이용건수 / 구별 문화누리대상자 추정인구.
- 접근성 기준선: 공공기관식 차량 10km 최근접 접근성만 사용함.
- 제외: H3SFCA, SFCA, 우리반경 접근성, 종합취약지수.
- 새 CSV나 이미지 파일은 저장하지 않고, 노트북 안에서 결과만 표시함.


## 01. 분석 환경 설정
- 원자료와 기존 전처리 산출물 경로를 지정함.
- 정부식 접근성은 `공공기관식_최근접접근성_*`, `공공기관식_서비스권역인구비율_*`만 사용함.


In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:,.4f}".format)

PROJECT = Path(r"C:\project\oracle_mnc_project")
MNC_CARD_PATH = PROJECT / "data" / "raw" / "mnc_card" / "mnc_seoul_usage_issuance_2021_2025.xlsx"
ANALYSIS_OUTPUT_PATH = PROJECT / "analysis_table" / "data" / "output"
PUBLIC_PATH = PROJECT / "notebooks" / "access" / "OUTPUT" / "public_access_index_25km"

GRID_POP_PATH = ANALYSIS_OUTPUT_PATH / "서울시_100m_문화누리추정인구수.gpkg"
AGE_GENDER_PATH = ANALYSIS_OUTPUT_PATH / "서울시_격자_100m_문화누리대상자_성연령별_인구수.csv"
DISABLED_PATH = ANALYSIS_OUTPUT_PATH / "서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv"
BASIC_PATH = ANALYSIS_OUTPUT_PATH / "서울시_기초생활수급자_행정동별_2023(가공).csv"
CLASSED_PATH = ANALYSIS_OUTPUT_PATH / "서울시_차상위계층_행정동별_2021_2023(가공).csv"
GOV_NEAREST_PATH = PUBLIC_PATH / "공공기관식_최근접접근성_서울시군구_중분류별.csv"
GOV_SERVICE_PATH = PUBLIC_PATH / "공공기관식_서비스권역인구비율_서울시군구_중분류별.csv"

for path in [
    MNC_CARD_PATH, GRID_POP_PATH, AGE_GENDER_PATH, DISABLED_PATH,
    BASIC_PATH, CLASSED_PATH, GOV_NEAREST_PATH, GOV_SERVICE_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(path)

print("PROJECT:", PROJECT)
print("정부식 접근성:", GOV_NEAREST_PATH.name)
print("정부식 서비스권역:", GOV_SERVICE_PATH.name)


## 02. 이용률 타깃 생성
- 2025년 문화누리카드 구별 이용건수 원자료에서 전체 이용건수와 중분류별 이용건수를 계산함.
- 분모는 서울 100m 격자 기반 구별 문화누리대상자 추정인구임.


In [ ]:
def clean_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )


def normalize_label(value):
    text = str(value).strip()
    for token in ["\n", " ", "\t", "\r", "(", ")", "ㆍ", "·"]:
        text = text.replace(token, "")
    return text


def find_usage_col(columns, label, unit="건"):
    target = normalize_label(f"{label}({unit})")
    match = [col for col in columns if normalize_label(col) == target]
    if not match:
        raise KeyError(f"이용건수 컬럼을 찾지 못함: {label}({unit})")
    return match[0]


middle_category_map = {
    "도서": ["도서"],
    "음악": ["음악"],
    "영상": ["영화", "TV"],
    "공연": ["공연"],
    "미술": ["전시", "공예", "사진관"],
    "문화체험": ["문화체험", "직업체험", "문화일반"],
    "관광지": ["관광명소", "휴양림/캠핑장", "동식물원", "온천", "체험관광", "테마파크"],
    "스포츠관람": ["스포츠관람"],
    "체육용품": ["체육용품"],
    "체육시설": ["체육시설"],
}

grid_pop = gpd.read_file(GRID_POP_PATH)
grid_pop = pd.DataFrame(grid_pop.drop(columns="geometry"))
for col in ["추정_인구수", "문화누리대상자_추정_인구수", "주택수", "공시지가"]:
    grid_pop[col] = pd.to_numeric(grid_pop[col], errors="coerce").fillna(0)

gu_pop = (
    grid_pop
    .groupby("시군구", as_index=False)
    .agg(
        구별_총추정인구=("추정_인구수", "sum"),
        구별_문화누리대상자추정인구=("문화누리대상자_추정_인구수", "sum"),
        구별_주택수=("주택수", "sum"),
        격자수=("GRID_CD", "nunique"),
    )
)
gu_pop["문화누리대상자비율"] = np.where(
    gu_pop["구별_총추정인구"] > 0,
    gu_pop["구별_문화누리대상자추정인구"] / gu_pop["구별_총추정인구"] * 100,
    np.nan,
)

raw = pd.read_excel(MNC_CARD_PATH, sheet_name="2025")
raw["광역"] = raw["광역"].astype(str).str.strip()
raw["기초"] = raw["기초"].astype(str).str.strip()
raw = raw[raw["광역"].eq("서울")].rename(columns={"기초": "시군구"}).copy()
raw["시군구"] = raw["시군구"].astype(str).str.strip()

overall_usage = raw[["시군구", "이용건수"]].copy()
overall_usage["이용건수"] = clean_number(overall_usage["이용건수"]).fillna(0)
overall_usage = overall_usage.merge(gu_pop, on="시군구", how="left")
overall_usage["대상자1인당_이용건수"] = np.where(
    overall_usage["구별_문화누리대상자추정인구"] > 0,
    overall_usage["이용건수"] / overall_usage["구별_문화누리대상자추정인구"],
    np.nan,
)
overall_usage["대상자천명당_이용건수"] = overall_usage["대상자1인당_이용건수"] * 1000

usage_rows = []
for mid, sub_labels in middle_category_map.items():
    cols = [find_usage_col(raw.columns, label, "건") for label in sub_labels]
    temp = raw[["시군구"]].copy()
    temp["중분류"] = mid
    temp["이용건수"] = sum(clean_number(raw[col]).fillna(0) for col in cols)
    usage_rows.append(temp)

category_usage = pd.concat(usage_rows, ignore_index=True)
category_usage = category_usage.merge(
    gu_pop[["시군구", "구별_문화누리대상자추정인구", "구별_총추정인구"]],
    on="시군구",
    how="left",
)
category_usage["대상자1인당_이용건수"] = np.where(
    category_usage["구별_문화누리대상자추정인구"] > 0,
    category_usage["이용건수"] / category_usage["구별_문화누리대상자추정인구"],
    np.nan,
)
category_usage["대상자천명당_이용건수"] = category_usage["대상자1인당_이용건수"] * 1000

print("전체 이용률 구조:", overall_usage.shape)
print("중분류 이용률 구조:", category_usage.shape)
print("시군구 수:", overall_usage["시군구"].nunique())
print("중분류 수:", category_usage["중분류"].nunique())

display(
    overall_usage[["시군구", "이용건수", "구별_문화누리대상자추정인구", "대상자천명당_이용건수"]]
    .sort_values("대상자천명당_이용건수", ascending=False)
    .head(8)
    .round(2)
)


## 03. 인구통계 특성 집계
- 격자별 문화누리대상자 추정인구를 구 단위로 합산함.
- 성별·연령대·장애 추정인구와 기초생활/차상위 구성 비율을 구 단위 변수로 만듦.


In [ ]:
def age_group(value):
    text = str(value)
    if text in ["0-5세", "6-14세", "15-19세"]:
        return "0-19세비율"
    if text in ["20-29세", "30-39세"]:
        return "20-39세비율"
    if text in ["40-49세", "50-59세"]:
        return "40-59세비율"
    return "60세이상비율"


age = pd.read_csv(
    AGE_GENDER_PATH,
    encoding="utf-8-sig",
    usecols=["시군구", "성별", "연령대", "문화누리대상자_성연령별_추정_인구수"],
)
age["문화누리대상자_성연령별_추정_인구수"] = pd.to_numeric(
    age["문화누리대상자_성연령별_추정_인구수"],
    errors="coerce",
).fillna(0)
age["연령그룹"] = age["연령대"].map(age_group)

age_total = (
    age.groupby("시군구", as_index=False)["문화누리대상자_성연령별_추정_인구수"]
    .sum()
    .rename(columns={"문화누리대상자_성연령별_추정_인구수": "성연령_대상자합"})
)
age_pivot = (
    age.groupby(["시군구", "연령그룹"], as_index=False)["문화누리대상자_성연령별_추정_인구수"]
    .sum()
    .pivot(index="시군구", columns="연령그룹", values="문화누리대상자_성연령별_추정_인구수")
    .reset_index()
)
age_pivot = age_pivot.merge(age_total, on="시군구", how="left")
for col in ["0-19세비율", "20-39세비율", "40-59세비율", "60세이상비율"]:
    age_pivot[col] = np.where(age_pivot["성연령_대상자합"] > 0, age_pivot[col] / age_pivot["성연령_대상자합"] * 100, np.nan)

gender = (
    age.groupby(["시군구", "성별"], as_index=False)["문화누리대상자_성연령별_추정_인구수"]
    .sum()
    .pivot(index="시군구", columns="성별", values="문화누리대상자_성연령별_추정_인구수")
    .reset_index()
)
gender["여성비율"] = np.where(
    (gender.get("남성", 0) + gender.get("여성", 0)) > 0,
    gender.get("여성", 0) / (gender.get("남성", 0) + gender.get("여성", 0)) * 100,
    np.nan,
)

disabled = pd.read_csv(
    DISABLED_PATH,
    encoding="utf-8-sig",
    usecols=["시군구", "문화누리대상자_성연령장애별_추정_인구수"],
)
disabled["문화누리대상자_성연령장애별_추정_인구수"] = pd.to_numeric(
    disabled["문화누리대상자_성연령장애별_추정_인구수"],
    errors="coerce",
).fillna(0)
disabled = (
    disabled.groupby("시군구", as_index=False)["문화누리대상자_성연령장애별_추정_인구수"]
    .sum()
    .rename(columns={"문화누리대상자_성연령장애별_추정_인구수": "장애추정대상자수"})
)

basic = pd.read_csv(BASIC_PATH, encoding="utf-8-sig").groupby("시군구", as_index=False)["기초생활수급자수"].sum()
classed = pd.read_csv(CLASSED_PATH, encoding="utf-8-sig").groupby("시군구", as_index=False)["차상위계층수급권자수"].sum()

demo = (
    gu_pop
    .merge(age_pivot[["시군구", "0-19세비율", "20-39세비율", "40-59세비율", "60세이상비율"]], on="시군구", how="left")
    .merge(gender[["시군구", "여성비율"]], on="시군구", how="left")
    .merge(disabled, on="시군구", how="left")
    .merge(basic, on="시군구", how="left")
    .merge(classed, on="시군구", how="left")
)
demo[["장애추정대상자수", "기초생활수급자수", "차상위계층수급권자수"]] = demo[
    ["장애추정대상자수", "기초생활수급자수", "차상위계층수급권자수"]
].fillna(0)
demo["장애추정비율"] = np.where(
    demo["구별_문화누리대상자추정인구"] > 0,
    demo["장애추정대상자수"] / demo["구별_문화누리대상자추정인구"] * 100,
    np.nan,
)
demo["기초생활비율"] = np.where(
    demo["구별_문화누리대상자추정인구"] > 0,
    demo["기초생활수급자수"] / demo["구별_문화누리대상자추정인구"] * 100,
    np.nan,
)
demo["차상위비율"] = np.where(
    demo["구별_문화누리대상자추정인구"] > 0,
    demo["차상위계층수급권자수"] / demo["구별_문화누리대상자추정인구"] * 100,
    np.nan,
)
demo["차상위대상자내비중"] = np.where(
    (demo["기초생활수급자수"] + demo["차상위계층수급권자수"]) > 0,
    demo["차상위계층수급권자수"] / (demo["기초생활수급자수"] + demo["차상위계층수급권자수"]) * 100,
    np.nan,
)

demo_cols = [
    "구별_문화누리대상자추정인구", "문화누리대상자비율", "기초생활비율", "차상위비율",
    "차상위대상자내비중", "여성비율", "0-19세비율", "20-39세비율", "40-59세비율",
    "60세이상비율", "장애추정비율",
]

demo_profile_cols = [
    "여성비율", "0-19세비율", "20-39세비율", "40-59세비율", "60세이상비율",
]

print("인구통계 테이블 구조:", demo.shape)
print("전체 생성 변수:", demo_cols)
print("상관 분석 사용 변수(성별·연령 비중):", demo_profile_cols)
display(demo[["시군구"] + demo_profile_cols].round(2).head())


## 04. 정부식 10km 접근성 결합
- 최근접 접근성은 거리가 짧을수록 접근성이 좋으므로 `-거리`로 방향을 맞춤.
- 서비스권역은 차량 10km 안에 들어오는 문화누리대상자 비율임.


In [ ]:
gov_nearest = pd.read_csv(GOV_NEAREST_PATH, encoding="utf-8-sig")
gov_service = pd.read_csv(GOV_SERVICE_PATH, encoding="utf-8-sig")

gov_access = gov_nearest.merge(
    gov_service[["시군구", "중분류", "문화누리대상자_서비스권역비율"]],
    on=["시군구", "중분류"],
    how="left",
)
gov_access["정부최근접_음거리"] = -pd.to_numeric(
    gov_access["문화누리대상자_가중평균_접근거리_m"],
    errors="coerce",
)
gov_access = gov_access.rename(columns={"문화누리대상자_서비스권역비율": "정부10km_서비스권역비율"})

demo_attach_cols = [col for col in demo_cols if col not in category_usage.columns]

category_panel = category_usage.merge(
    gov_access[["시군구", "중분류", "문화누리대상자_가중평균_접근거리_m", "정부최근접_음거리", "정부10km_서비스권역비율"]],
    on=["시군구", "중분류"],
    how="left",
).merge(demo[["시군구"] + demo_attach_cols], on="시군구", how="left")

print("정부식 접근성 결합 구조:", category_panel.shape)
print("정부 최근접 결측:", category_panel["정부최근접_음거리"].isna().sum())
print("서비스권역 결측:", category_panel["정부10km_서비스권역비율"].isna().sum())

display(
    gov_service
    .groupby("중분류", as_index=False)["문화누리대상자_서비스권역비율"]
    .agg(최소="min", 중앙값="median", 최대="max")
    .sort_values("중앙값")
    .round(2)
)


## 05. 상관 패턴 확인
- Pearson 상관 검정: 구별 이용률과 각 변수의 선형 관계가 0과 다른지 확인함.
- Williams 상관계수 차이 검정: 같은 25개 구에서 동일한 이용률을 공유하는 두 상관계수, 즉 대표 인구통계적 특성과 공공 문화접근성(최근접)의 상관계수가 서로 다른지 확인함.
- 대표 인구통계적 특성은 성별·연령 비중 변수 중 Pearson 상관계수의 절댓값이 가장 큰 변수 1개를 사용함.
- 그래프와 평균 비교는 관계의 강도를 보기 위해 절댓값 상관계수를 사용함. 단, 차이 검정은 부호가 있는 원래 Pearson r을 비교함.


In [ ]:
def pearson_result(df, x_col, y_col):
    temp = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(temp) < 3 or temp[x_col].nunique() < 2 or temp[y_col].nunique() < 2:
        return len(temp), np.nan, np.nan
    r, p = stats.pearsonr(temp[x_col], temp[y_col])
    return len(temp), r, p


def sig_label(p):
    if pd.isna(p):
        return "검정불가"
    return "유의함" if p < 0.05 else "유의하지 않음"


def williams_corr_diff(df, y_col, x1_col, x2_col):
    temp = df[[y_col, x1_col, x2_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(temp) < 4:
        return len(temp), np.nan, np.nan, np.nan
    if temp[y_col].nunique() < 2 or temp[x1_col].nunique() < 2 or temp[x2_col].nunique() < 2:
        return len(temp), np.nan, np.nan, np.nan

    n = len(temp)
    r_yx1 = stats.pearsonr(temp[y_col], temp[x1_col])[0]
    r_yx2 = stats.pearsonr(temp[y_col], temp[x2_col])[0]
    r_x1x2 = stats.pearsonr(temp[x1_col], temp[x2_col])[0]

    k = 1 - r_yx1**2 - r_yx2**2 - r_x1x2**2 + 2 * r_yx1 * r_yx2 * r_x1x2
    denom = 2 * k * (n - 1) / (n - 3) + ((r_yx1 + r_yx2) ** 2 / 4) * (1 - r_x1x2) ** 3
    numerator_term = (n - 1) * (1 + r_x1x2)

    if denom <= 0 or numerator_term <= 0:
        return n, r_yx1, r_yx2, np.nan

    t_stat = (r_yx1 - r_yx2) * np.sqrt(numerator_term / denom)
    p = stats.t.sf(abs(t_stat), df=n - 3) * 2
    return n, r_yx1, r_yx2, p


overall_demo_attach_cols = [col for col in demo_cols if col not in overall_usage.columns]
overall_demo = overall_usage.merge(demo[["시군구"] + overall_demo_attach_cols], on="시군구", how="left")

overall_corr_rows = []
for col in demo_profile_cols:
    n, r, p = pearson_result(overall_demo, col, "대상자1인당_이용건수")
    overall_corr_rows.append({
        "변수": col,
        "n": n,
        "Pearson": r,
        "p값": p,
        "유의성": sig_label(p),
        "abs_r": abs(r) if pd.notna(r) else np.nan,
    })
overall_corr = pd.DataFrame(overall_corr_rows).sort_values("abs_r", ascending=False)

gov_corr_rows = []
for mid, group in category_panel.groupby("중분류"):
    n, r, p = pearson_result(group, "정부최근접_음거리", "대상자1인당_이용건수")
    gov_corr_rows.append({
        "중분류": mid,
        "지표": "공공 문화접근성(최근접)",
        "n": n,
        "Pearson": r,
        "p값": p,
        "유의성": sig_label(p),
        "abs_r": abs(r) if pd.notna(r) else np.nan,
    })
gov_corr = pd.DataFrame(gov_corr_rows).sort_values("abs_r", ascending=False)

cat_demo_rows = []
for mid, group in category_panel.groupby("중분류"):
    for col in demo_profile_cols:
        n, r, p = pearson_result(group, col, "대상자1인당_이용건수")
        cat_demo_rows.append({
            "중분류": mid,
            "변수": col,
            "n": n,
            "Pearson": r,
            "p값": p,
            "유의성": sig_label(p),
            "abs_r": abs(r) if pd.notna(r) else np.nan,
        })
cat_demo_corr = pd.DataFrame(cat_demo_rows)
cat_demo_top = (
    cat_demo_corr.sort_values(["중분류", "abs_r"], ascending=[True, False])
    .groupby("중분류", as_index=False)
    .head(1)
    .sort_values("abs_r", ascending=False)
)

compare_rows = []
for mid in sorted(category_panel["중분류"].dropna().unique()):
    access = gov_corr[gov_corr["중분류"].eq(mid)].sort_values("abs_r", ascending=False).head(1)
    profile = cat_demo_corr[cat_demo_corr["중분류"].eq(mid)].sort_values("abs_r", ascending=False).head(1)
    if access.empty or profile.empty:
        continue

    group = category_panel[category_panel["중분류"].eq(mid)]
    profile_col = profile["변수"].iloc[0]
    n_diff, profile_r_for_test, gov_r_for_test, diff_p = williams_corr_diff(
        group,
        "대상자1인당_이용건수",
        profile_col,
        "정부최근접_음거리",
    )

    access_abs = abs(gov_r_for_test) if pd.notna(gov_r_for_test) else float(access["abs_r"].iloc[0])
    profile_abs = abs(profile_r_for_test) if pd.notna(profile_r_for_test) else float(profile["abs_r"].iloc[0])
    if profile_abs - access_abs >= 0.10:
        pattern = "인구특성 쪽이 더 큼"
    elif access_abs - profile_abs >= 0.10:
        pattern = "공공 문화접근성 쪽이 더 큼"
    else:
        pattern = "비슷함"

    compare_rows.append({
        "중분류": mid,
        "인구특성변수": profile_col,
        "인구특성_r": profile_r_for_test,
        "정부최근접_r": gov_r_for_test,
        "상관차이_p값": diff_p,
        "상관차이_유의성": sig_label(diff_p),
        "정부식변수": access["지표"].iloc[0],
        "정부식_abs_r": access_abs,
        "인구특성_abs_r": profile_abs,
        "차이_abs_r": profile_abs - access_abs,
        "잠정패턴": pattern,
    })
compare_table = pd.DataFrame(compare_rows)
compare_table["상관차이_유의순위"] = compare_table["상관차이_유의성"].eq("유의함").astype(int)
compare_table = (
    compare_table
    .sort_values(["상관차이_유의순위", "인구특성_abs_r"], ascending=[False, False])
    .drop(columns="상관차이_유의순위")
)

mean_abs_corr = pd.DataFrame([
    {"지표": "인구통계적 특성", "평균 |Pearson r|": compare_table["인구특성_abs_r"].mean()},
    {"지표": "공공 문화접근성(최근접)", "평균 |Pearson r|": compare_table["정부식_abs_r"].mean()},
])

cat_demo_detail_top5 = (
    cat_demo_corr
    .sort_values(["중분류", "abs_r"], ascending=[True, False])
    .groupby("중분류", as_index=False)
    .head(5)
    .sort_values(["중분류", "abs_r"], ascending=[True, False])
)

cat_demo_matrix = (
    cat_demo_corr
    .pivot(index="중분류", columns="변수", values="Pearson")
    .reindex(columns=demo_profile_cols)
    .round(3)
)

print("전체 이용률 vs 인구특성 비중")
display(overall_corr.drop(columns="abs_r").round(4))

print("중분류 이용률 vs 공공 문화접근성(최근접)")
display(gov_corr.drop(columns="abs_r").round(4))

print("중분류별 가장 관계가 큰 인구특성 비중")
display(cat_demo_top[["중분류", "변수", "Pearson", "p값", "유의성"]].round(4))

print("중분류별 인구특성 비중 상관관계 상세: 상위 5개")
display(cat_demo_detail_top5[["중분류", "변수", "Pearson", "p값", "유의성"]].round(4))

print("중분류 x 인구특성 비중 Pearson 상관계수 매트릭스")
display(cat_demo_matrix)

print("공공 문화접근성(최근접) vs 대표 인구통계적 특성 비교")
compare_display = (
    compare_table[[
        "중분류", "인구특성변수", "인구특성_r", "정부최근접_r",
        "상관차이_p값", "상관차이_유의성", "잠정패턴",
    ]]
    .rename(columns={
        "인구특성변수": "인구통계특성",
        "인구특성_r": "인구통계_r",
        "정부최근접_r": "최근접_r",
        "상관차이_p값": "p값",
        "상관차이_유의성": "유의성",
    })
)
display(compare_display.round(4))

print("평균 절댓값 Pearson 상관계수")
display(mean_abs_corr.round(4))


## 06. 시각화: 인구통계적 특성 vs 최근접 접근성
- 각 중분류마다 성별·연령 비중 중 대표 인구통계적 특성 1개와 공공 문화접근성(최근접) 1개를 비교함.
- 막대 길이는 관계 강도를 보기 위해 절댓값 Pearson r로 표시하고, 막대 끝 수치는 부호가 있는 원래 Pearson r임.
- 보조 지도는 정부 10km 서비스권역비율이 구 단위에서 대부분 높게 나타나는지 확인하기 위한 지도임.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch
from matplotlib.cm import ScalarMappable

STYLE = {
    "background": "#fbf6ef",
    "text": "#241f1c",
    "axis": "#d9d3cb",
    "tick": "#766f67",
    "comparison": "#c7c2bc",
    "orange": "#f26b30",
    "teal": "#009688",
}


def configure_project_font():
    font_source = Path(r"C:\Windows\Fonts\NotoSansKR-VF.ttf")
    if font_source.exists():
        try:
            import tempfile
            from fontTools.ttLib import TTFont
            from fontTools.varLib import instancer

            for weight in [400, 900]:
                font_path = Path(tempfile.gettempdir()) / f"NotoSansKR-{weight}.ttf"
                if not font_path.exists():
                    font = instancer.instantiateVariableFont(TTFont(str(font_source)), {"wght": weight})
                    font.save(str(font_path))
                font_manager.fontManager.addfont(str(font_path))

            plt.rcParams["font.family"] = "Noto Sans KR"
            plt.rcParams["axes.unicode_minus"] = False
            return
        except Exception:
            pass

    installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in ["Noto Sans KR", "Malgun Gothic", "Apple SD Gothic Neo", "AppleGothic", "DejaVu Sans"]:
        if font_name in installed_fonts:
            plt.rcParams["font.family"] = font_name
            break
    else:
        plt.rcParams["font.family"] = "DejaVu Sans"
    plt.rcParams["axes.unicode_minus"] = False


def label_bar(axis, value, y_position, signed_value, color=STYLE["text"]):
    if pd.isna(value):
        return
    axis.text(
        value + 0.010,
        y_position,
        f"{signed_value:.2f}",
        va="center",
        ha="left",
        fontsize=11.5,
        color=color,
    )


configure_project_font()

plot_data = compare_table.reset_index(drop=True)
max_value = plot_data[["인구특성_abs_r", "정부식_abs_r"]].max().max()
xmax = max(0.62, max_value + 0.075)

fig = plt.figure(figsize=(11.0, 8.1), dpi=180)
fig.patch.set_facecolor(STYLE["background"])
gs = fig.add_gridspec(1, 2, width_ratios=[4.2, 1.18], wspace=0.025, left=0.13, right=0.965, bottom=0.08, top=0.685)
axis = fig.add_subplot(gs[0, 0])
label_axis = fig.add_subplot(gs[0, 1], sharey=axis)

axis.set_facecolor(STYLE["background"])
label_axis.set_facecolor(STYLE["background"])

y_positions = np.arange(len(plot_data))
bar_height = 0.27

axis.barh(
    y_positions - bar_height / 2,
    plot_data["인구특성_abs_r"],
    height=bar_height,
    color=STYLE["orange"],
    edgecolor="none",
)
axis.barh(
    y_positions + bar_height / 2,
    plot_data["정부식_abs_r"],
    height=bar_height,
    color=STYLE["comparison"],
    edgecolor="none",
)

for y_position, (_, row) in zip(y_positions, plot_data.iterrows()):
    label_bar(axis, row["인구특성_abs_r"], y_position - bar_height / 2, row["인구특성_r"])
    label_bar(axis, row["정부식_abs_r"], y_position + bar_height / 2, row["정부최근접_r"], STYLE["tick"])
    label_axis.text(
        0.02,
        y_position - bar_height / 2,
        row["인구특성변수"],
        va="center",
        ha="left",
        fontsize=11.5,
        color=STYLE["text"],
    )

axis.axvline(0, color=STYLE["axis"], linewidth=1.15, zorder=0)
axis.set_xlim(0, xmax)
axis.set_yticks(y_positions)
axis.set_yticklabels(plot_data["중분류"], fontsize=13.2, color=STYLE["text"])
axis.invert_yaxis()
axis.set_xlabel("상관계수 절댓값 |Pearson r|", fontsize=11.5, color=STYLE["tick"], labelpad=9)
axis.grid(False)
axis.tick_params(axis="x", labelsize=10.5, colors=STYLE["tick"], length=5, width=1)
axis.tick_params(axis="y", length=0, pad=9)

for spine in ["top", "right", "left"]:
    axis.spines[spine].set_visible(False)
axis.spines["bottom"].set_color(STYLE["axis"])
axis.spines["bottom"].set_linewidth(1.0)

label_axis.set_xlim(0, 1)
label_axis.set_xticks([])
label_axis.tick_params(axis="y", left=False, labelleft=False)
for spine in ["top", "right", "bottom", "left"]:
    label_axis.spines[spine].set_visible(False)

legend_handles = [
    Patch(facecolor=STYLE["orange"], edgecolor="none", label="인구통계적 특성"),
    Patch(facecolor=STYLE["comparison"], edgecolor="none", label="공공 문화접근성"),
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.835),
    ncol=2,
    frameon=False,
    fontsize=12.2,
)
fig.suptitle(
    "인구통계적 특성 vs 최근접 접근성: 이용률 상관계수 비교",
    fontsize=21.5,
    fontweight=900,
    color=STYLE["text"],
    y=0.965,
)

plt.show()

gu_boundary_path = PROJECT / "analysis_table" / "data" / "input" / "raw" / "spatial" / "boundary" / "seoul_gu_boundary.json"
gu_boundary = gpd.read_file(gu_boundary_path)
service_map_data = (
    gov_service
    .groupby("시군구", as_index=False)["문화누리대상자_서비스권역비율"]
    .agg(정부10km_서비스권역중앙값="median", 정부10km_서비스권역최소="min")
)
service_map = gu_boundary.merge(service_map_data, left_on="name", right_on="시군구", how="left")

cmap = LinearSegmentedColormap.from_list(
    "service_teal_orange",
    ["#1b9aaa", "#57c7b6", "#f2d16b", "#f79a3e", "#d94f2b"],
)
norm = Normalize(vmin=90, vmax=100)

fig, axis = plt.subplots(figsize=(7.2, 5.35), dpi=180)
fig.patch.set_facecolor(STYLE["background"])
axis.set_facecolor(STYLE["background"])

service_map.plot(
    column="정부10km_서비스권역중앙값",
    cmap=cmap,
    norm=norm,
    ax=axis,
    edgecolor=STYLE["background"],
    linewidth=0.75,
    missing_kwds={"color": "#e7ded4", "edgecolor": STYLE["background"]},
)

for _, row in service_map.iterrows():
    point = row.geometry.representative_point()
    label = row["name"].replace("구", "")
    axis.text(
        point.x,
        point.y,
        label,
        ha="center",
        va="center",
        fontsize=6.8,
        fontweight=900,
        color=STYLE["text"],
        bbox={
            "boxstyle": "round,pad=0.10",
            "facecolor": STYLE["background"],
            "edgecolor": "none",
            "alpha": 0.64,
        },
    )

axis.set_axis_off()
axis.set_title(
    "공공 문화접근성 10km 서비스권역비율 지도",
    fontsize=17,
    fontweight=900,
    color=STYLE["text"],
    pad=13,
)

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axis, fraction=0.035, pad=0.015)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=8, colors=STYLE["tick"], length=0)
cbar.set_label("구별 중앙값(%)", fontsize=8.5, color=STYLE["tick"], labelpad=8)

plt.tight_layout()
plt.show()


## 07. 주요 해석
- 아래 해석은 상관 패턴을 보는 EDA이며, 인과관계 검정은 아님.
- Pearson 상관은 구별 이용률과 각 변수의 선형 관계 방향을 확인하기 위해 사용함.


In [ ]:
service_summary = (
    gov_service
    .groupby("중분류", as_index=False)["문화누리대상자_서비스권역비율"]
    .agg(중앙값="median", 최소="min")
    .sort_values("중앙값")
)
low_service = service_summary.iloc[0]

significant_overall = overall_corr[overall_corr["p값"] < 0.05]["변수"].tolist()
significant_gov = gov_corr[gov_corr["p값"] < 0.05][["중분류", "지표", "Pearson"]]
significant_diff = compare_table[compare_table["상관차이_p값"] < 0.05][["중분류", "인구특성변수"]]
pattern_counts = compare_table["잠정패턴"].value_counts().to_dict()

top_overall = overall_corr.iloc[0]
mean_profile = mean_abs_corr.loc[mean_abs_corr["지표"].eq("인구통계적 특성"), "평균 |Pearson r|"].iloc[0]
mean_gov = mean_abs_corr.loc[mean_abs_corr["지표"].eq("공공 문화접근성(최근접)"), "평균 |Pearson r|"].iloc[0]

lines = [
    "### 결과 요약",
    "- 인구특성 상관분석에서는 구별 대상자 규모와 문화누리대상자비율을 제외하고, 성별·연령 구성 비중만 사용함.",
    "- Pearson 상관 검정은 각 변수와 이용률의 관계가 0과 다른지 확인하기 위해 사용함.",
    "- Williams 상관계수 차이 검정은 동일한 구별 이용률에 대해 인구통계적 특성의 상관계수와 공공 문화접근성(최근접)의 상관계수가 서로 다른지 확인하기 위해 사용함.",
    f"- 정부식 10km 서비스권역비율은 `{low_service['중분류']}`가 가장 낮고, 중앙값은 {low_service['중앙값']:.2f}%임.",
    f"- 나머지 대부분 중분류는 중앙값이 99% 이상이라, 서비스권역비율만으로는 구별 차이를 잘 가르기 어려움.",
    f"- 전체 이용률과 가장 크게 움직인 인구통계적 특성은 `{top_overall['변수']}`이며 Pearson r={top_overall['Pearson']:.3f}, p={top_overall['p값']:.4f}임.",
    f"- 공공 문화접근성(최근접)과 인구통계적 특성 비교에서는 `{pattern_counts.get('인구특성 쪽이 더 큼', 0)}`개 중분류가 인구특성 쪽 관계가 더 컸고, `{pattern_counts.get('공공 문화접근성 쪽이 더 큼', 0)}`개 중분류가 공공 문화접근성 쪽 관계가 더 컸음.",
    f"- 평균 절댓값 Pearson r은 인구통계적 특성 {mean_profile:.3f}, 공공 문화접근성(최근접) {mean_gov:.3f}임.",
]

if significant_overall:
    lines.append("- 전체 이용률 기준 유의한 인구통계적 특성: " + ", ".join(f"`{v}`" for v in significant_overall))
else:
    lines.append("- 전체 이용률 기준 p<0.05인 인구통계적 특성은 없음.")

if len(significant_gov) > 0:
    formatted = [
        f"`{row.중분류}`-{row.지표}(r={row.Pearson:.3f})"
        for row in significant_gov.itertuples(index=False)
    ]
    lines.append("- 중분류 이용률 기준 유의한 공공 문화접근성(최근접) 관계: " + ", ".join(formatted))
else:
    lines.append("- 중분류 이용률 기준 p<0.05인 공공 문화접근성(최근접) 관계는 없음.")

if len(significant_diff) > 0:
    formatted = [f"`{row.중분류}`({row.인구특성변수})" for row in significant_diff.itertuples(index=False)]
    lines.append("- 두 상관계수 차이가 유의한 중분류: " + ", ".join(formatted))
else:
    lines.append("- 두 상관계수 차이가 p<0.05 기준 유의한 중분류는 없음.")

lines.append("- 따라서 앞단 문제정의에서는 `정부식 10km 접근성만으로 이용률 차이를 충분히 설명하기 어렵다`는 문제 제기가 가능함.")
lines.append("- 다음 단계 가설은 `구별 이용률은 단순 시설 접근성보다 성별·연령 구성 같은 인구 특성과 더 관련될 수 있다`로 두는 것이 자연스러움.")

display(Markdown("\n".join(lines)))
